### 1. Import AgentPy

In [2]:
# Run this cell first to import the library

%pip install agentpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.1/780.1 kB 2.1 MB/s  0:00:00 eta 0:00:01
  Attempting uninstall: dill
    Found existing installation: dill 0.3.8
    Uninstalling dill-0.3.8:
      Successfully uninstalled dill-0.3.8
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [agentpy]m2/4 [SALib]rocess]

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
import agentpy as ap

### 2.The MachineAgent

In [5]:
class MachineAgent(ap.Agent):

    # The setup method acts as our constructor in AgentPy
    def setup(self, capability="Drilling"):
        # Save the machine's capability
        self.capability = capability

        # Initialize an empty list to act as our message inbox
        self.inbox = []

        # Print a startup message confirming the agent's ID and capability
        print(f"Machine Agent {self.id} is ready. Registered capability: {self.capability}")

### 3. The OrderAgent

In [6]:
class OrderAgent(ap.Agent):

    def setup(self, required_operation="Drilling"):
        # Save what operation this order needs
        self.required_operation = required_operation

        # Initialize its own empty inbox
        self.inbox = []

        # Print a startup message
        print(f"Order Agent {self.id} is ready. Required operation: {self.required_operation}")

#### 4. Testing the Agents

In [7]:
# 1. Create a blank base model to house our agents
dummy_model = ap.Model()

# 2. Spawn one MachineAgent and pass the capability argument
machine1 = MachineAgent(dummy_model, capability="Drilling")

# 3. Spawn one OrderAgent and pass the required_operation argument
order1 = OrderAgent(dummy_model, required_operation="Drilling")

Machine Agent 1 is ready. Registered capability: Drilling
Order Agent 2 is ready. Required operation: Drilling


## Phase 2 - Building the Model Environment

### This is where AgentPy really simplifies things compared to JADE. Instead of relying on a complex Directory Facilitator (the Yellow Pages) to discover other agents, AgentPy uses a centralized Model to orchestrate everything. The Model acts as the environment that holds all the agents together. This means that any agent can easily access any other agent through the Model, without needing to go through a separate discovery process. The Model provides a straightforward way for agents to find and interact with each other, making the implementation much simpler and more intuitive.

### 5. The Model Environment

In [8]:
class ManufacturingModel(ap.Model):

    def setup(self):
        # 1. Instantiate agents using ap.AgentList
        # This creates lists containing 1 agent of each type, passing the required arguments
        self.machines = ap.AgentList(self, 1, MachineAgent, capability="Drilling")
        self.orders = ap.AgentList(self, 1, OrderAgent, required_operation="Drilling")

        # 2. Service Discovery: Give the OrderAgent its "Contact List"
        # We grab the first (and only) order agent at index 0 and hand it the list of machines
        self.orders[0].target_machines = self.machines

        print("\n--- Model Setup Complete ---")

# 3. Execution Block
if __name__ == '__main__':
    # Instantiate the model
    model = ManufacturingModel()

    # Run the setup to spawn the agents and trigger their print statements
    model.setup()

Machine Agent 1 is ready. Registered capability: Drilling
Order Agent 2 is ready. Required operation: Drilling

--- Model Setup Complete ---


## Phase 3 - Enabling Agent Communication

### By using simple state variables (like self.state), we can easily control whether an agent does something every single tick (like listening) or just once (like sending an initial request).

In [9]:
class MachineAgent(ap.Agent):

    def setup(self, capability="Drilling"):
        self.capability = capability
        self.inbox = []
        print(f"Machine Agent {self.id} is ready. Registered capability: {self.capability}")

    # --- PHASE 3: The Step Method ---
    def step(self):
        # Because there is no state check here, this will print on EVERY tick
        print(f"Machine Agent {self.id} is waiting for work...")

### updated OrderAgent with a state variable to control when it sends its request

In [10]:
class OrderAgent(ap.Agent):

    def setup(self, required_operation="Drilling"):
        self.required_operation = required_operation
        self.inbox = []

        # --- PHASE 3: State Tracking ---
        self.state = 'seeking_machine'

        print(f"Order Agent {self.id} is ready. Required operation: {self.required_operation}")

    # --- PHASE 3: The Step Method ---
    def step(self):
        # This acts like a JADE OneShotBehaviour because we change the state immediately after!
        if self.state == 'seeking_machine':
            print(f"Order Agent {self.id} is looking for machines...")

            # Change the state so this block never runs again
            self.state = 'waiting_for_bids'